In [ ]:
# Import required libraries
import kagglehub # for importing the dataset
import pandas as pd # dataframe stuff
import numpy as np # in general, it's a mathematical library
import torch # admin this lab
import torch.nn as nn # for building the model
from torch.optim import AdamW # optimizer
import matplotlib.pyplot as plt # visualizing
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}") # checking for GPU

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

import pandas as pd

data_path = f"{path}/Q3_data.csv"
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head()



In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:

df.isnull().sum()

In [ ]:
#Do we have missing values?

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 4: Write your code here:

#Standardize features using StandardScaler (DON'T scale the target!)


numerical_cols =  df.select_dtypes(include=["number"]).columns.drop("Target")

scaler = StandardScaler()

# Apply fit_transform to scale the numerical columns
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()



In [ ]:
from sklearn.preprocessing import StandardScaler

# Apply feature scaling to numerical features

target_column_name = 'Target'

if target_column_name in df.columns:
    numerical_features = df.drop(columns=[target_column_name]).select_dtypes(include=['number']).columns
    print(f"Excluding target column '{target_column_name}' from scaling.")
else:
    numerical_features = df.select_dtypes(include=['number']).columns
    print(f"Warning: Target column '{target_column_name}' not found. All numerical columns are being scaled.")
    print("If your target column has a different name, please replace 'target' in the code above.")

scaler = StandardScaler()
df[numerical_features] = scaler.fit_transform(df[numerical_features])


In [ ]:
# Task 5: Write your code here:
df['Target'].value_counts()

In [ ]:
# Task 1: Write your code here:
X_train, X_test = X.iloc[train_index], X.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]

In [ ]:
target_column_name = 'Target'

# Check if the target column exists
if target_column_name not in df.columns:
    raise ValueError(f"Target column '{target_column_name}' not found in the DataFrame. Please check the column name.")

X = df.drop(columns=[target_column_name])
y = df[target_column_name]

print("Dataset split into features (X) and target (y).")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:

# Stratified 5-Fold Cross-Validation, shuffled
n_splits = 5 # K
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


# Train a CatBoostClassifier model
sklearn_model = {"CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
  }

In [ ]:
##### برجع له لاني لخبطت فيه

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import numpy as np

# Check for target imbalance
print("\nTarget distribution:")
print(y.value_counts(normalize=True))

if y.value_counts(normalize=True).min() < 0.2:
    print("Target is imbalanced, using StratifiedKFold.")
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    metric_to_use = 'f1_score'
else:
    print("Target is balanced, using KFold.")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    metric_to_use = 'accuracy'

# Initialize lists to store scores
fold_scores = []

# Train the CatBoostClassifier model with cross-validation
print(f"\nTraining CatBoostClassifier with {kf.n_splits}-fold {type(kf).__name__}...")

for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3,
        loss_function='Logloss',
        eval_metric='F1' if metric_to_use == 'f1_score' else 'Accuracy',
        random_seed=42,
        verbose=0,
        early_stopping_rounds=50
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=0)

    y_pred = model.predict(X_val)

    if metric_to_use == 'f1_score':
        score = f1_score(y_val, y_pred)
    else:
        score = model.get_best_score()['validation']['Accuracy']

    fold_scores.append(score)
    print(f"Fold {fold+1} {metric_to_use}: {score:.4f}")

# Print the averaged score
average_score = np.mean(fold_scores)
print(f"\nAverage {metric_to_use} across all folds: {average_score:.4f}")

In [ ]:
# Task 1: Write your code here:


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: